In [117]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [118]:
df = pd.read_csv('qoute_dataset.csv')

In [119]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [120]:
df.shape

(3038, 2)

In [121]:
quotes = df['quote']
quotes

0       “The world as we have created it is a process ...
1       “It is our choices, Harry, that show what we t...
2       “There are only two ways to live your life. On...
3       “The person, be it gentleman or lady, who has ...
4       “Imperfection is beauty, madness is genius and...
                              ...                        
3033        The past beats inside me like a second heart.
3034    Damn, Claire. Warn a guy before you do a face-...
3035    Can you be a girl for a few seconds?""I'm alwa...
3036    That's what fiction is for. It's for getting a...
3037    If we have no peace, it is because we have for...
Name: quote, Length: 3038, dtype: str

In [122]:
quotes = quotes.str.lower()
quotes 

0       “the world as we have created it is a process ...
1       “it is our choices, harry, that show what we t...
2       “there are only two ways to live your life. on...
3       “the person, be it gentleman or lady, who has ...
4       “imperfection is beauty, madness is genius and...
                              ...                        
3033        the past beats inside me like a second heart.
3034    damn, claire. warn a guy before you do a face-...
3035    can you be a girl for a few seconds?""i'm alwa...
3036    that's what fiction is for. it's for getting a...
3037    if we have no peace, it is because we have for...
Name: quote, Length: 3038, dtype: str

In [123]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
quotes 

0       “the world as we have created it is a process ...
1       “it is our choices harry that show what we tru...
2       “there are only two ways to live your life one...
3       “the person be it gentleman or lady who has no...
4       “imperfection is beauty madness is genius and ...
                              ...                        
3033         the past beats inside me like a second heart
3034    damn claire warn a guy before you do a facepla...
3035    can you be a girl for a few secondsim always a...
3036    thats what fiction is for its for getting at t...
3037    if we have no peace it is because we have forg...
Name: quote, Length: 3038, dtype: str

In [124]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence  import pad_sequences

In [125]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)
sequences = tokenizer.texts_to_sequences(quotes)

In [126]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [127]:

X = []
y = []

for seq in sequences:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [128]:
len(X)

85271

In [129]:
len(y)

85271

In [ ]:
maxlen = max(len(s) for s in X)
X_padded = pad_sequences(X, maxlen=maxlen, padding='pre')
X_padded

array([[   0,    0,    0, ...,  752,   70, 2461],
       [   0,    0,    0, ...,   54,   70, 3676],
       [   0,    0,    0, ...,    7,    5, 3677],
       ...,
       [   0,    0,    0, ...,   26, 2411, 8978],
       [   0,    0,    0, ...,   17,    1,  174],
       [   0,    0,    0, ...,    3,  169,  101]],
      shape=(3038, 746), dtype=int32)

In [ ]:
y = np.array(y)
print('Number of training samples:', len(X))
print('Shape of padded inputs:', X_padded.shape)
print('Shape of labels:', y.shape)

In [ ]:
from tensorflow.keras.utils import to_categorical
one_hot_encoded_y = to_categorical(y, num_classes=vocab_size)

In [133]:
one_hot_encoded_y.shape

(85271, 10000)

In [134]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [135]:
embedding_dim = 50
rnn_units = 128

In [136]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen,name='embed')
)

rnn_model.add(
    SimpleRNN(units=rnn_units, name='simple_rnn')
)

rnn_model.add(
    Dense(
        vocab_size, activation='softmax'
    )
)

In [137]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accurancy']
)

In [138]:
rnn_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [139]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen,name='embed')
)

lstm_model.add(
    LSTM(units=rnn_units, name='lstm')
)

lstm_model.add(
    Dense(
        vocab_size, activation='softmax'
    )
)

In [140]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accurancy']
)

In [141]:
lstm_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [142]:
epochs=10
batch_size=128

In [145]:
rnn_model.fit(
    X_padded, 
    one_hot_encoded_y, 
    epochs=epochs, 
    batch_size=batch_size, 
    verbose=1,
    validation_split=0.1
    )

Epoch 1/10


ValueError: Could not interpret metric identifier: accurancy

In [ ]:
epochs=100
batch_size=128

In [146]:
lstm_model.fit(
    X_padded, 
    one_hot_encoded_y, 
    epochs=epochs, 
    batch_size=batch_size, 
    verbose=1)

ValueError: Data cardinality is ambiguous. Make sure all arrays contain the same number of samples.'x' sizes: 3038
'y' sizes: 85271
